In [1]:
# 매치정보를 통해서 PALYER_ID를 가져오는 함수
import requests
import json
import os

# 매치 데이터를 요청하는 함수
def get_match_data(match_id):
    print(f"Retrieving match data for match_id: {match_id}...")  # 매치 데이터 요청 메시지
    headers = {
        'accept': 'application/vnd.api+json',
    }
    response = requests.get('https://api.pubg.com/shards/steam/matches/' + match_id, headers=headers)
    if response.status_code == 200:
        print(f"Successfully retrieved data for match_id: {match_id}")
        return response.json()
    else:
        print(f"Failed to retrieve data for match_id: {match_id} (status code: {response.status_code})")
        return None

# 플레이어의 정보(PLAYER_ID 등)를 추출하는 함수
def extract_player_info(match_data):
    if match_data is None:
        return []

    # 매치 데이터에서 참가자 정보 추출
    participants = match_data['included']
    player_info_list = []

    for participant in participants:
        # 참가자가 플레이어 정보인지 확인 (type이 'participant'인지 체크)
        if participant['type'] == 'participant':
            player_info = {
                'player_id': participant['attributes']['stats']['playerId']
            }
            player_info_list.append(player_info)

    return player_info_list

# 플레이어 정보를 JSON 파일에 추가하는 함수
def append_player_info_to_json(player_info_list, filename):
    # 파일이 없으면 빈 리스트 생성
    if not os.path.exists(filename):
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump([], f)

    # 기존 파일에 새로운 플레이어 정보를 추가
    try:
        with open(filename, 'r+', encoding='utf-8') as f:
            # 기존 데이터를 읽어온 뒤
            data = json.load(f)
            # 새로운 데이터를 추가
            data.extend(player_info_list)
            # 파일의 시작으로 돌아가서 덮어쓰기
            f.seek(0)
            json.dump(data, f, ensure_ascii=False, indent=4)
        print(f"Player info successfully appended to {filename}")
    except Exception as e:
        print(f"Error appending player info to JSON: {e}")

# 매치 ID를 입력받아 플레이어 정보를 추출하고 파일에 추가
match_id = '4d222aea-355c-4703-8ea4-e427600f7ca3'  # 여기에 매치 ID 입력
output_file_name = "TEST.json"  # 고정된 출력 파일명
match_data = get_match_data(match_id)

if match_data:
    player_info_list = extract_player_info(match_data)
    if player_info_list:
        append_player_info_to_json(player_info_list, output_file_name)
    else:
        print("No player info found.")
else:
    print("No match data available.")


Retrieving match data for match_id: 4d222aea-355c-4703-8ea4-e427600f7ca3...
Successfully retrieved data for match_id: 4d222aea-355c-4703-8ea4-e427600f7ca3
Player info successfully appended to TEST.json


In [ ]:
import requests
import json
import time

# API 요청 헤더 (인증 없이 사용하려면 Authorization 부분 제거)
headers = {
    'accept': 'application/vnd.api+json',
    'Authorization': 'Bearer s',
}

# 유저 목록 파일 경로 (player_id 리스트가 저장된 JSON 파일)
user_file_path = 'TEST.json'

# 매치 정보 저장 파일 경로
output_file_path = 'clean_user_list.json'

# 유저 정보를 담을 리스트
users = []

# JSON 파일에서 유저 정보 읽기
with open(user_file_path, 'r') as file:
    users = json.load(file)

# 기존 저장된 매치 정보가 있다면 불러오기
try:
    with open(output_file_path, 'r') as output_file:
        match_info = json.load(output_file)
        existing_user_ids = {user['id'] for user in match_info}  # 이미 저장된 유저들의 player_id를 집합(set)으로 저장
except FileNotFoundError:
    match_info = []  # 파일이 없으면 빈 리스트로 시작
    existing_user_ids = set()

# 10명씩 나누어 처리
batch_size = 10
total_users = len(users)  # 처리할 유저 수

for i in range(0, total_users, batch_size):
    batch_users = [user for user in users[i:i + batch_size] if not user['player_id'].startswith('ai')]  # 'ai'로 시작하는 player_id는 제외
    player_ids = ",".join([user['player_id'] for user in batch_users])
    
    # batch_users가 비어있으면 다음으로 넘어감
    if not player_ids:
        continue

    url = f'https://api.pubg.com/shards/steam/players?filter[playerIds]={player_ids}'

    try:
        # API 호출
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            # 응답 데이터를 JSON으로 변환
            match_data = response.json()

            # 각 플레이어에 대해 처리
            for user in batch_users:
                player_data = next((item for item in match_data.get('data', []) if item['id'] == user['player_id']), None)
                if player_data:
                    ban_type = player_data.get('attributes', {}).get('banType', 'Innocent')
                    print(f"User: {user['player_id']}, BanType: {ban_type}")
                    
                    if ban_type == 'Innocent':  # 밴 상태가 'Innocent'인 경우에만 추출
                        # 유저가 이미 존재하는지 확인
                        if user['player_id'] not in existing_user_ids:
                            match_info.append({
                                'id': user['player_id'],
                                'matches': player_data  # 추가로 플레이어의 상세 정보를 저장
                            })
                            existing_user_ids.add(user['player_id'])  # 새롭게 추가된 유저 player_id를 집합에 추가

                            # 매번 새로운 유저 정보를 즉시 JSON 파일에 저장
                            with open(output_file_path, 'w') as output_file:
                                json.dump(match_info, output_file, indent=4)

        else:
            print(f"Failed to fetch data for users {player_ids}, Status code: {response.status_code}")
        
        # 진행 상황 표시
        print(f"Processing users {i + 1} to {i + len(batch_users)}/{total_users}")

        # 대기 시간 (API rate limit을 피하기 위해)
        time.sleep(6)

    except Exception as e:
        print(f"An error occurred while processing users {player_ids}: {e}")

print(f"Filtered match information has been saved to {output_file_path}.")


User: account.c01e05723f73480a86898b9782581069, BanType: Innocent
User: account.8012c55b75874adb86a8cad0460f3300, BanType: Innocent
User: account.ae4fcb5af3fe44cb9261e89aee4c34c4, BanType: Innocent
User: account.6451e97cf5d64b53a0e8b3846c29418c, BanType: Innocent
User: account.282d1a22a2c04d34af6f2622c718eb83, BanType: Innocent
User: account.e6468212744e4b06b04b2814aa58b949, BanType: Innocent
User: account.21fe2df5bbe546ea8728060501f4c9bb, BanType: Innocent
User: account.fa8114bbab534b4c97dc2717f9270cff, BanType: Innocent
User: account.71615fee035e40a8a0d977fa40c7bf46, BanType: Innocent
User: account.de4647b7dafd45e88be7f1607491a63f, BanType: Innocent
Processing users 1 to 10/191
User: account.1a4c815b25874827ae3832ceef103aab, BanType: Innocent
User: account.0495a5ab6cd44ee7ac74d4b1886e6908, BanType: Innocent
User: account.2ef1f4f5edbc4972a543bc622fa13da1, BanType: Innocent
User: account.a5becf442392449ba20f0a56c28e0d16, BanType: Innocent
User: account.f9ba2e5ac353453dafbfb3a13d5dc798,

In [45]:
# 리더보드에서의 JSON의 COL값을 추출하기 위해서 사용
# Rank500등의 id, name, rank정보를 추출한다.
import json

def extract_json_to_json(json_file_path, output_json_file_path):
    # JSON 파일 읽기
    with open(json_file_path, 'r', encoding='utf-8') as json_file:
        data = json.load(json_file)
    
    # 추출할 데이터를 저장할 리스트
    extracted_data = []

    # JSON 데이터에서 필요한 정보 추출
    for player in data['included']:
        player_info = {
            "id": player["id"],
            "name": player["attributes"]["name"],
            "rank": player["attributes"]["rank"]
        }
        extracted_data.append(player_info)

    # 추출된 데이터를 새로운 JSON 파일로 저장
    with open(output_json_file_path, 'w', encoding='utf-8') as output_file:
        json.dump(extracted_data, output_file, ensure_ascii=False, indent=4)

# JSON 파일 경로 및 출력 JSON 파일 경로 설정
#json_file_path = 'illegal_player_data.json'
json_file_path = 'leaderboard_PUBG.json'  # 원본 JSON 파일 경로
output_json_file_path = 'HOIISSHIT.json'  # 출력할 JSON 파일 경로

# 함수 실행: JSON 데이터에서 ID, NAME, RANK 추출 후 JSON으로 저장
extract_json_to_json(json_file_path, output_json_file_path)

print(f"추출된 데이터가 저장된 JSON 파일: {output_json_file_path}")

추출된 데이터가 저장된 JSON 파일: HOIISSHIT.json


In [ ]:
import requests
import json
import time
import os

headers = {
    'accept': 'application/vnd.api+json',
}

# API를 사용하여 MATCH 정보를 가져오는 함수
def get_match_data(match_id):
    print(f"Retrieving match data for match_id: {match_id}...")  # 매치 데이터 요청 메시지
    response = requests.get('https://api.pubg.com/shards/steam/matches/' + match_id, headers=headers)
    if response.status_code == 200:
        print(f"Successfully retrieved data for match_id: {match_id}")
        return response.json()
    else:
        print(f"Failed to retrieve data for match_id: {match_id} (status code: {response.status_code})")
        return None
    time.sleep(0.1)  # API 과도한 사용 방지

# JSON 데이터에서 사용자 ID와 매치 ID 추출하는 함수
def load_match_ids_and_accounts(json_data):
    match_ids = []
    account_ids = []
    
    # 각 플레이어 데이터를 순회
    for player in json_data:
        account_id = player['id']  # 사용자 ID 추출
        account_ids.append(account_id)
        
        # 'relationships' -> 'matches' -> 'data' 경로로 매치 데이터 접근
        if 'matches' in player['matches']['relationships'] and 'data' in player['matches']['relationships']['matches']:
            matches_data = player['matches']['relationships']['matches']['data']
            
            # 각 매치의 ID를 추출
            for match in matches_data:
                match_ids.append(match['id'])
    
    print(f"Total matches found: {len(match_ids)}")  # 매치 ID의 수 출력
    return match_ids, account_ids

# 이미 처리된 match_id를 로드하는 함수
def load_processed_match_ids():
    if os.path.exists('processed_matches.json'):
        with open('processed_matches.json', 'r') as file:
            return json.load(file)
    return []

# 처리된 match_id를 저장하는 함수
def save_processed_match_ids(processed_match_ids):
    with open('processed_matches.json', 'w') as file:
        json.dump(processed_match_ids, file, indent=4)

# 특정 accountId에 맞는 참가자 데이터만 저장하는 함수
def save_filtered_match_data(match, participants, account_ids):
    game_mode = match['data']['attributes']['gameMode']
    
    if game_mode == 'solo' or game_mode == 'solo-fpp':
        file_name = 'solo_data3.json'
    elif game_mode == 'duo' or game_mode == 'duo-fpp':
        file_name = 'duo_data3.json'
    elif game_mode == 'squad' or game_mode == 'squad-fpp':
        file_name = 'squad_data3.json'
    else:
        print(f"Unknown game mode for match: {match['data']['id']}")
        return
    
    # 파일이 존재하는지 확인하고 데이터를 로드
    if os.path.exists(file_name):
        if os.stat(file_name).st_size != 0:
            with open(file_name, 'r') as file:
                try:
                    data = json.load(file)
                except json.JSONDecodeError:
                    print(f"Error reading {file_name}. The file might be corrupted or invalid.")
                    data = []
        else:
            data = []
    else:
        data = []

    # accountId와 일치하는 참가자 데이터만 필터링 (playerId를 사용)
    filtered_participants = [
        p for p in participants
        if p['playerId'] in account_ids and p['damageDealt'] > 0 and p['timeSurvived'] >= 240
    ]

    # 매치와 필터링된 참가자 데이터를 저장
    if filtered_participants:
        match_info = {
            "match_id": match['data']['id'],
            "game_mode": game_mode,
            "match_data": match['data']['attributes'],
            "participants": filtered_participants
        }

        # 데이터를 파일에 추가
        data.append(match_info)

        # 파일에 저장
        with open(file_name, 'w') as file:
            json.dump(data, file, indent=4)
        
        print(f"Filtered data for match {match['data']['id']} saved to {file_name}")
    else:
        print(f"No valid participants found for match {match['data']['id']}")

# 특정 accountId의 매치 데이터를 처리하는 함수 (중복 처리 방지 기능 추가)
def process_matches_for_accounts(match_ids, account_ids):
    # 이미 처리된 match_id를 로드
    processed_match_ids = load_processed_match_ids()
    
    # 새롭게 처리할 match_id 리스트
    new_match_ids = [match_id for match_id in match_ids if match_id not in processed_match_ids]
    
    if not new_match_ids:
        print("No new matches to process.")
        return

    for idx, match_id in enumerate(new_match_ids, start=1):
        print(f"Processing match {idx}/{len(new_match_ids)} (match_id: {match_id})...")
        match_data = get_match_data(match_id)
        if match_data:
            # 매치 데이터에서 참가자 추출
            participants = [item['attributes']['stats'] for item in match_data['included'] if item['type'] == 'participant']
            # 특정 accountId에 맞는 데이터를 필터링하여 저장
            save_filtered_match_data(match_data, participants, account_ids)
            # 처리된 match_id 추가
            processed_match_ids.append(match_id)
            # 처리된 match_id를 실시간으로 저장
            save_processed_match_ids(processed_match_ids)

# JSON 파일을 불러오고 처리 (파일에서 불러오는 경우)
file_path = 'ban_user_list.json'
with open(file_path, 'r') as file:
    json_data = json.load(file)

# match_id와 account_id 추출 및 처리
match_ids, account_ids = load_match_ids_and_accounts(json_data)
process_matches_for_accounts(match_ids, account_ids)


In [4]:
import requests
import json
import time
import os

headers = {
    'accept': 'application/vnd.api+json',
}

# 이미 처리된 match_id를 로드하는 함수
def load_processed_match_ids():
    if os.path.exists('processed_matches.json'):
        with open('processed_matches.json', 'r') as file:
            return json.load(file)
    return []

# 처리된 match_id를 저장하는 함수
def save_processed_match_ids(processed_match_ids):
    with open('processed_matches.json', 'w') as file:
        json.dump(processed_match_ids, file, indent=4)

# API를 사용하여 MATCH 정보를 가져온다.
def get_match_data(match_id):
    print(f"Retrieving match data for match_id: {match_id}...")  # 매치 데이터 요청 메시지
    response = requests.get('https://api.pubg.com/shards/steam/matches/' + match_id, headers=headers)
    if response.status_code == 200:
        print(f"Successfully retrieved data for match_id: {match_id}")
        return response.json()
    else:
        print(f"Failed to retrieve data for match_id: {match_id} (status code: {response.status_code})")
        return None
    time.sleep(0.1)  # API의 과도한 사용 방지

# JSON 데이터에서 match ID를 추출하는 함수
def load_match_ids(json_data):
    match_ids = []
    
    # 각 플레이어 데이터를 순회
    for player in json_data:
        # 'matches' -> 'relationships' -> 'matches' -> 'data'에서 match 데이터를 가져오기
        if 'matches' in player and 'relationships' in player['matches']:
            matches_data = player['matches']['relationships']['matches']['data']
            
            # 각 매치의 ID를 추출
            for match in matches_data:
                match_ids.append(match['id'])
    
    print(f"Total matches found: {len(match_ids)}")  # 매치 ID의 수 출력
    return match_ids

# 매치 데이터를 게임 모드에 따라 파일에 저장
def save_match_and_participant_data(match, participants):
    game_mode = match['data']['attributes']['gameMode']
    # 스쿼드 매치만 추출하는 경우
    if game_mode == 'squad' or game_mode == 'squad-fpp':
        file_name = 'new_normal_squad_data15.json'
    else :
        print(f"Unknown game mode for match: {match['data']['id']}")
        return
    # 솔로, 듀오, 스쿼드 데이터를 모두 추출하는 경우
    '''
    if game_mode == 'solo' or game_mode == 'solo-fpp':
        file_name = 'normal_solo_data9.json'
    elif game_mode == 'duo' or game_mode == 'duo-fpp':
        file_name = 'normal_duo_data9.json'
    elif game_mode == 'squad' or game_mode == 'squad-fpp':
        file_name = 'normal_squad_data9.json'
    else:
        print(f"Unknown game mode for match: {match['data']['id']}")
        return
    '''
    # 파일이 존재하는지 확인하고 데이터를 로드
    if os.path.exists(file_name):
        if os.stat(file_name).st_size != 0:
            with open(file_name, 'r') as file:
                try:
                    data = json.load(file)
                except json.JSONDecodeError:
                    print(f"Error reading {file_name}. The file might be corrupted or invalid.")
                    data = []
        else:
            data = []
    else:
        data = []

    # 매치와 참가자 데이터를 함께 저장
    match_info = {
        "match_id": match['data']['id'],
        "game_mode": game_mode,
        "match_data": match['data']['attributes'],
        "participants": participants
    }

    # 데이터를 기존 리스트에 추가
    data.append(match_info)

    # 기존 데이터를 유지하면서 파일에 append
    with open(file_name, 'w') as file:
        json.dump(data, file, indent=4)
    
    print(f"Data for match {match['data']['id']} saved to {file_name}")

# 매치 데이터를 처리하고 저장 (1000개 매치 처리 후 종료)
def process_matches(match_ids, max_matches=1000):
    processed_match_ids = load_processed_match_ids()  # 이미 처리된 match_id 로드
    match_count = 0  # 처리된 매치 카운트

    for idx, match_id in enumerate(match_ids, start=1):
        # 이미 처리된 match_id는 건너뛰기
        if match_id in processed_match_ids:
            print(f"Skipping already processed match {match_id}")
            continue

        # 1000개 매치를 처리했을 때 함수 종료
        if match_count >= max_matches:
            print(f"Processed {max_matches} matches. Stopping.")
            break

        print(f"Processing match {idx}/{len(match_ids)}...")  # 현재 매치 진행 상황 출력
        match_data = get_match_data(match_id)
        if match_data:
            # 'participant' 데이터 추출
            participants = [item['attributes']['stats'] for item in match_data['included'] if item['type'] == 'participant']

            # damageDealt > 0, timeSurvived >= 240 필터 적용
            filtered_participants = [
                p for p in participants
                if p['damageDealt'] > 0 and p['timeSurvived'] >= 240
            ]

            if filtered_participants:  # 필터링된 참가자가 있을 때만 저장
                save_match_and_participant_data(match_data, filtered_participants)

            # 처리된 match_id 리스트에 추가하고 즉시 저장
            processed_match_ids.append(match_id)
            save_processed_match_ids(processed_match_ids)  # 처리된 매치 ID 즉시 저장
            print(f"Progress saved after match {match_id}")
            print(f"현재 처리한 매치: {match_count}")
            match_count += 1  # 매치 카운트 증가

# JSON 파일을 불러오고 처리 (파일에서 불러오는 경우)
file_path = 'clean_user_list.json'
with open(file_path, 'r') as file:
    json_data = json.load(file)

# match_id 추출 및 처리
match_ids = load_match_ids(json_data)

# 매치 처리 함수 호출 (최대 N개 매치 처리)
process_matches(match_ids, max_matches=800) # N은 처리할 매치 수, 원하는 만큼 입력


Total matches found: 26086
Skipping already processed match d9d0e6e0-456b-4266-9acf-9e79dea493f0
Skipping already processed match 6627dd59-3eb6-41f8-aca7-748249b48dd0
Skipping already processed match eec022ec-1f28-4444-996f-2beceaa40a34
Skipping already processed match b5ba4cb4-f19f-4a50-b05e-1cec0de2106c
Skipping already processed match 4e629b02-9676-4c4d-b789-2d6ca73d216e
Skipping already processed match 1fc3e056-4180-4ba7-b914-56396f76815e
Skipping already processed match 6e49b477-f157-4aa2-af3e-8a2f9e2e0700
Skipping already processed match 5b81875a-bd56-4585-a82d-a974c13ea9d9
Skipping already processed match 70b01120-3c8b-4ba4-af60-7c3bd7086890
Skipping already processed match 9c30584d-ecbc-4823-ac39-d18178bc585d
Skipping already processed match 71516dd2-477e-4719-8762-cb702b16fcc7
Skipping already processed match 4313c7a0-f06e-4480-a589-84ef8dfece8c
Skipping already processed match 8dd45193-04dd-48e5-8869-a2f4118b720a
Skipping already processed match 420d215a-ba93-4102-a5b8-9a3915

In [ ]:
# 매치에서 특정한 유저의 아이디만을 추출하기 위한 함수
import requests
import json
import time

headers = {
    'accept': 'application/vnd.api+json',
}

# API를 사용하여 MATCH 정보를 가져오는 함수
def get_match_data(match_id):
    print(f"Retrieving match data for match_id: {match_id}...")  # 매치 데이터 요청 메시지
    response = requests.get('https://api.pubg.com/shards/steam/matches/' + match_id, headers=headers)
    if response.status_code == 200:
        print(f"Successfully retrieved data for match_id: {match_id}")
        return response.json()
    else:
        print(f"Failed to retrieve data for match_id: {match_id} (status code: {response.status_code})")
        return None
    time.sleep(0.1)  # API 과도한 사용 방지

# JSON 데이터에서 사용자 ID와 매치 ID 추출하는 함수
def load_match_ids_and_accounts(json_data):
    match_ids = []
    account_ids = []
    
    # 각 플레이어 데이터를 순회
    for player in json_data:
        account_id = player['id']  # 사용자 ID 추출
        account_ids.append(account_id)
        
        # 'relationships' -> 'matches' -> 'data' 경로로 매치 데이터 접근
        if 'matches' in player['matches']['relationships'] and 'data' in player['matches']['relationships']['matches']:
            matches_data = player['matches']['relationships']['matches']['data']
            
            # 각 매치의 ID를 추출
            for match in matches_data:
                match_ids.append(match['id'])
    
    print(f"Total matches found: {len(match_ids)}")  # 매치 ID의 수 출력
    return match_ids, account_ids

# 이미 처리된 match_id를 로드하는 함수
def load_processed_match_ids():
    if os.path.exists('processed_matches.json'):
        with open('processed_matches.json', 'r') as file:
            return json.load(file)
    return []

# 처리된 match_id를 저장하는 함수
def save_processed_match_ids(processed_match_ids):
    with open('processed_matches.json', 'w') as file:
        json.dump(processed_match_ids, file, indent=4)

# 특정 accountId에 맞는 참가자 데이터만 저장하는 함수
def save_filtered_match_data(match, participants, account_ids):
    game_mode = match['data']['attributes']['gameMode']
    
    if game_mode == 'solo' or game_mode == 'solo-fpp':
        file_name = 'solo_data3.json'
    elif game_mode == 'duo' or game_mode == 'duo-fpp':
        file_name = 'duo_data3.json'
    elif game_mode == 'squad' or game_mode == 'squad-fpp':
        file_name = 'squad_data3.json'
    else:
        print(f"Unknown game mode for match: {match['data']['id']}")
        return
    
    # 파일이 존재하는지 확인하고 데이터를 로드
    if os.path.exists(file_name):
        if os.stat(file_name).st_size != 0:
            with open(file_name, 'r') as file:
                try:
                    data = json.load(file)
                except json.JSONDecodeError:
                    print(f"Error reading {file_name}. The file might be corrupted or invalid.")
                    data = []
        else:
            data = []
    else:
        data = []

    # accountId와 일치하는 참가자 데이터만 필터링 (playerId를 사용)
    filtered_participants = [
        p for p in participants
        if p['playerId'] in account_ids and p['damageDealt'] > 0 and p['timeSurvived'] >= 240
    ]

    # 매치와 필터링된 참가자 데이터를 저장
    if filtered_participants:
        match_info = {
            "match_id": match['data']['id'],
            "game_mode": game_mode,
            "match_data": match['data']['attributes'],
            "participants": filtered_participants
        }

        # 데이터를 파일에 추가
        data.append(match_info)

        # 기존 데이터에 추가하는 방식으로 파일에 저장 (덮어쓰지 않음)
        with open(file_name, 'w') as file:
            json.dump(data, file, indent=4)
        
        print(f"Filtered data for match {match['data']['id']} saved to {file_name}")
    else:
        print(f"No valid participants found for match {match['data']['id']}")

# 특정 accountId의 매치 데이터를 처리하는 함수 (중복 처리 방지 기능 추가)
def process_matches_for_accounts(match_ids, account_ids):
    # 이미 처리된 match_id를 로드
    processed_match_ids = load_processed_match_ids()
    
    # 새롭게 처리할 match_id 리스트
    new_match_ids = [match_id for match_id in match_ids if match_id not in processed_match_ids]
    
    if not new_match_ids:
        print("No new matches to process.")
        return

    for idx, match_id in enumerate(new_match_ids, start=1):
        print(f"Processing match {idx}/{len(new_match_ids)} (match_id: {match_id})...")
        match_data = get_match_data(match_id)
        if match_data:
            # 매치 데이터에서 참가자 추출
            participants = [item['attributes']['stats'] for item in match_data['included'] if item['type'] == 'participant']
            # 특정 accountId에 맞는 데이터를 필터링하여 저장
            save_filtered_match_data(match_data, participants, account_ids)
            # 처리된 match_id 추가
            processed_match_ids.append(match_id)
            # 처리된 match_id를 실시간으로 저장
            save_processed_match_ids(processed_match_ids)

# JSON 파일을 불러오고 처리 (파일에서 불러오는 경우)
file_path = 'ban_user_list.json'
with open(file_path, 'r') as file:
    json_data = json.load(file)

# match_id와 account_id 추출 및 처리
match_ids, account_ids = load_match_ids_and_accounts(json_data)
process_matches_for_accounts(match_ids, account_ids)


In [1]:
import json

all_data = []

# JSON 파일을 리스트로 병합
for i in range(1, 15):
    with open(f"new_normal_squad_data{i}.json", encoding='UTF-8') as file:
        data = json.load(file)
        if isinstance(data, list):
            all_data.extend(data)  # 리스트일 경우 확장
        elif isinstance(data, dict):
            all_data.append(data)  # 딕셔너리일 경우 추가

print("엑")

# 병합된 데이터를 JSON 파일로 저장
with open("new_PUBG_player_squad.json", 'w', encoding='UTF-8') as file:
    json.dump(all_data, file, ensure_ascii=False, indent=4)


엑


In [2]:
import json
"""
all_data = []

with open("PUBG_ban_player_squad.json", encoding = 'UTF-8') as file:
    data = json.load(file)
    if isinstance(data, list):
        all_data.extend(data)
    elif  isinstance(data, dict):
        all_data.append(data)

with open("PUBG_player_squad.json", encoding = 'UTF-8') as file:
    data = json.load(file)
    if isinstance(data, list):
        all_data.extend(data)
    elif  isinstance(data, dict):
        all_data.append(data)

with open("SHITOFSHIT.json", 'w', encoding = 'UTF-8') as file:
    json.dump(all_data, file, ensure_ascii=False, indent=4)
"""


all_data = []

# 첫 번째 파일: PUBG_ban_player_squad.json
with open("new_PUBG_player_squad.json", encoding='UTF-8') as file:
    data = json.load(file)
    if isinstance(data, list):
        for item in data:
            # 파일 구분을 위해 'ban_' 접두어를 붙임
            item = {f"ban_{key}" if key in ['damageDealt'] else key: value for key, value in item.items()}
            all_data.append(item)
    elif isinstance(data, dict):
        item = {f"ban_{key}" if key in ['damageDealt'] else key: value for key, value in data.items()}
        all_data.append(item)

# 두 번째 파일: PUBG_player_squad.json
with open("PUBG_player_squad.json", encoding='UTF-8') as file:
    data = json.load(file)
    if isinstance(data, list):
        for item in data:
            # 파일 구분을 위해 'player_' 접두어를 붙임
            item = {f"player_{key}" if key in ['damageDealt'] else key: value for key, value in item.items()}
            all_data.append(item)
    elif isinstance(data, dict):
        item = {f"player_{key}" if key in ['damageDealt'] else key: value for key, value in data.items()}
        all_data.append(item)

# 병합된 데이터를 새로운 JSON 파일로 저장
with open("PUBG_player_squad.json", 'w', encoding='UTF-8') as file:
    json.dump(all_data, file, ensure_ascii=False, indent=4)
